In [1]:
from google.colab import files

uploaded = files.upload()

Saving corrected_medication_logs.csv to corrected_medication_logs (3).csv


In [2]:
from google.colab import files

uploaded = files.upload()

Saving diet_logs.csv to diet_logs (3).csv


In [18]:
from google.colab import files

uploaded = files.upload()

Saving tasks.csv to tasks (6).csv


In [4]:
from google.colab import files

uploaded = files.upload()

Saving symptoms.csv to symptoms (3).csv


In [5]:
from google.colab import files

uploaded = files.upload()

Saving workout-logs-month-corrected.csv to workout-logs-month-corrected (3).csv


In [6]:
from google.colab import files

uploaded = files.upload()

Saving patients.csv to patients (3).csv


In [7]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

In [19]:
# Load datasets
med_logs = pd.read_csv('corrected_medication_logs.csv')
diet_logs = pd.read_csv('diet_logs.csv')
workout_logs = pd.read_csv('workout-logs-month-corrected.csv')
tasks = pd.read_csv('tasks (6).csv')
symptoms = pd.read_csv('symptoms.csv')
patients = pd.read_csv('patients.csv')

In [20]:
# Helper functions
def adherence_rate(df, patientId, status_col, taken_value='taken', missed_value='missed'):
    df_p = df[df['patientId'] == patientId]
    total = len(df_p)
    if total == 0:
        return np.nan, 0
    taken = (df_p[status_col] == taken_value).sum()
    missed = (df_p[status_col] == missed_value).sum()
    return taken / total * 100, missed

def diet_adherence(df, patientId):
    df_p = df[df['patientId'] == patientId]
    total = len(df_p)
    if total == 0:
        return np.nan
    consumed = (df_p['status'] == 'consumed').sum()
    return consumed / total * 100

def workout_adherence(df, patientId):
    df_p = df[df['patientId'] == patientId]
    total = len(df_p)
    if total == 0:
        return np.nan
    completed = (df_p['status'] == 'completed').sum()
    return completed / total * 100

def task_adherence(df, patientId):
    df_p = df[df['assignedTo'] == patientId]
    total = len(df_p)
    if total == 0:
        return np.nan
    completed = (df_p['status'] == 'completed').sum()
    return completed / total * 100

def risk_from_symptoms(df, patientId):
    df_p = df[df['patientId'] == patientId]
    if df_p.empty:
        return 0
    # Example: assign 2 for severe, 1 for moderate, 0.5 for mild
    severity_map = {'severe': 2, 'moderate': 1, 'mild': 0.5}
    return df_p['severity'].map(severity_map).sum()

def last_updated(df, patientId, date_col):
    df_p = df[df['patientId'] == patientId]
    if df_p.empty:
        return ""
    return df_p[date_col].max()

In [21]:
# Generate metrics for each patient
metrics = []
for pid in patients['patientId']:
    # Medication adherence
    med_adherence, missed_doses = adherence_rate(med_logs, pid, 'status', taken_value='taken', missed_value='missed')
    # Diet adherence
    diet_adher = diet_adherence(diet_logs, pid)
    # Workout adherence
    workout_adher = workout_adherence(workout_logs, pid)
    # Task adherence
    task_adher = task_adherence(tasks, pid)
    # Symptom risk
    risk_symptoms = risk_from_symptoms(symptoms, pid)
    # Risk: missed critical meds (example, here all missed are counted as general)
    risk_missedCriticalMeds = 0  # Placeholder, needs med classification
    risk_missedGeneralMeds = missed_doses * 0.2  # Example weight
    risk_alerts = 0.2 if missed_doses > 0 else 0
    # Total risk (sum of risk components as example)
    totalRisk = risk_symptoms + risk_missedCriticalMeds + risk_missedGeneralMeds + risk_alerts
    # Overall score (sum of adherence scores, can be weighted)
    scores = [x for x in [med_adherence, diet_adher, workout_adher, task_adher] if not np.isnan(x)]
    overallScore = sum(scores)
    # Last updated (latest from med logs)
    lastUpdated = last_updated(med_logs, pid, 'logged_at')
    # Workout streak (max consecutive completed)
    df_w = workout_logs[workout_logs['patientId'] == pid].sort_values('date')
    streak = 0
    max_streak = 0
    for status in df_w['status']:
        if status == 'completed':
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    metrics.append({
        "patientId": pid,
        "overall_adheranceScore": round(overallScore, 2),
        "medication_adheranceScore": round(med_adherence, 2) if not np.isnan(med_adherence) else 0.0,
        "tasks_adheranceScore": round(task_adher, 2) if not np.isnan(task_adher) else 0.0,
        "workouts_adheranceScore": round(workout_adher, 2) if not np.isnan(workout_adher) else 0.0,
        "diet_adheranceScore": round(diet_adher, 2) if not np.isnan(diet_adher) else 0.0,
        "totalRisk": round(totalRisk, 2),
        "risk_symptoms": round(risk_symptoms, 2),
        "risk_missedCriticalMeds": round(risk_missedCriticalMeds, 2),
        "risk_missedGeneralMeds": round(risk_missedGeneralMeds, 2),
        "risk_alerts": round(risk_alerts, 2),
        "missedDoses": int(missed_doses),
        "workoutStreak": int(max_streak),
        "lastUpdated": lastUpdated
    })

In [22]:
# Export to CSV
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv('patient_metrics.csv', index=False)

# Export to JSON
with open('patient_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Patient metrics generated in patient_metrics.csv and patient_metrics.json")

Patient metrics generated in patient_metrics.csv and patient_metrics.json


In [24]:
from google.colab import files

files.download('patient_metrics.csv')
files.download('patient_metrics.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
# Load and preprocess as before
med_logs = pd.read_csv('corrected_medication_logs.csv')
med_logs['scheduled_time'] = pd.to_datetime(med_logs['scheduled_time'], errors='coerce')
med_logs['taken_time'] = pd.to_datetime(med_logs['taken_time'], errors='coerce')
med_logs['logged_at'] = pd.to_datetime(med_logs['logged_at'], errors='coerce')

diet_logs = pd.read_csv('diet_logs.csv')
diet_logs['date'] = pd.to_datetime(diet_logs['date'], format='%d-%m-%Y', errors='coerce')

workout_logs = pd.read_csv('workout-logs-month-corrected.csv')
workout_logs['date'] = pd.to_datetime(workout_logs['date'], format='%d-%m-%Y', errors='coerce')

# Medication alerts
med_alerts_missed = med_logs[med_logs['status'] == 'missed'].copy()
med_alerts_missed['alert_type'] = 'Missed Dose'
med_alerts_severe = med_logs[med_logs['side_effects__severity'] >= 3].copy()
med_alerts_severe['alert_type'] = 'Severe Side Effect'

med_alerts_missed = med_alerts_missed.rename(columns={'log_id': 'alert_id', 'logged_at': 'alert_datetime'})
med_alerts_severe = med_alerts_severe.rename(columns={'log_id': 'alert_id', 'logged_at': 'alert_datetime'})
med_alerts_missed['source'] = 'medication'
med_alerts_severe['source'] = 'medication'
med_alerts_missed['alert_datetime'] = med_alerts_missed['alert_datetime'].dt.tz_localize(None)
med_alerts_severe['alert_datetime'] = med_alerts_severe['alert_datetime'].dt.tz_localize(None)
med_alerts_missed = med_alerts_missed[['alert_id', 'patientId', 'alert_type', 'source', 'alert_datetime', 'reason_missed', 'side_effects__description', 'side_effects__severity', 'side_effects__notes']]
med_alerts_severe = med_alerts_severe[['alert_id', 'patientId', 'alert_type', 'source', 'alert_datetime', 'reason_missed', 'side_effects__description', 'side_effects__severity', 'side_effects__notes']]

# Diet alerts
diet_alerts = diet_logs[diet_logs['status'] == 'skipped'].copy()
diet_alerts['alert_type'] = 'Skipped Meal'
diet_alerts['source'] = 'diet'
diet_alerts = diet_alerts.rename(columns={'log_id': 'alert_id', 'date': 'alert_datetime'})
diet_alerts['alert_datetime'] = pd.to_datetime(diet_alerts['alert_datetime'], errors='coerce')
diet_alerts = diet_alerts[['alert_id', 'patientId', 'alert_type', 'source', 'alert_datetime', 'diet_plan', 'meal_time']]

# Workout alerts
workout_alerts = workout_logs[workout_logs['status'] == 'missed'].copy()
workout_alerts['alert_type'] = 'Missed Workout'
workout_alerts['source'] = 'workout'
workout_alerts = workout_alerts.rename(columns={'log_id': 'alert_id', 'date': 'alert_datetime'})
workout_alerts['alert_datetime'] = pd.to_datetime(workout_alerts['alert_datetime'], errors='coerce')
workout_alerts = workout_alerts[['alert_id', 'patientId', 'alert_type', 'source', 'alert_datetime', 'workoutId', 'feedback']]

# Combine and sort
alerts = pd.concat([med_alerts_missed, med_alerts_severe, diet_alerts, workout_alerts], ignore_index=True)
alerts['alert_datetime'] = pd.to_datetime(alerts['alert_datetime'], errors='coerce')
alerts = alerts.sort_values(by=['patientId', 'alert_datetime'])

# Export to CSV
alerts.to_csv('alerts_table.csv', index=False)

# Convert datetimes to ISO strings for JSON
alerts_json = alerts.copy()
alerts_json['alert_datetime'] = alerts_json['alert_datetime'].dt.strftime('%Y-%m-%dT%H:%M:%S')
alerts_json = alerts_json.where(pd.notnull(alerts_json), None)
with open('alerts_table.json', 'w') as f:
    json.dump(alerts_json.to_dict(orient='records'), f, indent=2)

